# Ridge Extraction and Reconstruction

This notebook demonstrates wavesst's ridge-based signal analysis tools:

1. **Standard ridge extraction and reconstruction** — two crossing chirps, compare reconstructed vs original
2. **Component onset detection** — gated tone, `detect_onsets()` with energy thresholding
3. **Masked ridge extraction** — peel-off approach to find secondary components
4. **Parallel band-decomposed extraction** — compare `extract_ridges_parallel` vs sequential

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wavesst

cfg = wavesst.Config(device='cpu', dtype='complex128')
print(f'wavesst {wavesst.__version__}')

## Section 1: Standard Ridge Extraction and Reconstruction

Two chirps crossing in the time-frequency plane are a classic challenge for ridge extraction:
- **Chirp A**: linear 20 → 80 Hz
- **Chirp B**: quadratic 80 → 20 Hz

We synthesize them, mix them, run SST, extract two ridges, and reconstruct each component.

In [ ]:
FS = 256.0
T  = 2.0
t  = np.arange(int(T * FS)) / FS

# Linear chirp: 20 -> 80 Hz
chirp1 = wavesst.make_chirp(f_start=20.0, f_end=80.0, duration=T, fs=FS)

# Quadratic chirp: 80 -> 20 Hz  (reverse the frequency axis)
chirp2 = wavesst.make_chirp(f_start=80.0, f_end=20.0, duration=T, fs=FS, chirp_type='quadratic')

x = chirp1 + chirp2
print(f'Signal length: {len(x)} samples  ({T}s at {FS} Hz)')

In [ ]:
result = wavesst.sst(x, fs=FS, nv=32, gamma='auto', cfg=cfg)
ridges = wavesst.extract_ridges(result, n=2, penalty=1.0)

for i, r in enumerate(ridges):
    print(f'Ridge {i+1}: freq range {r.freq_path.min():.1f} – {r.freq_path.max():.1f} Hz, '
          f'median {np.median(r.freq_path):.1f} Hz, energy {r.energy:.4f}')

In [ ]:
comps = wavesst.reconstruct(result, ridges)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

originals = [chirp1, chirp2]
labels    = ['Chirp A (20→80 Hz)', 'Chirp B (80→20 Hz)']

for ax, orig, comp, lbl in zip(axes, originals, comps, labels):
    orig_np = orig if isinstance(orig, np.ndarray) else orig.numpy()
    comp_np = comp if isinstance(comp, np.ndarray) else comp.numpy()
    ax.plot(t, orig_np, color='steelblue', linewidth=1.0, label='Original')
    ax.plot(t, comp_np, color='tomato',    linewidth=0.8, linestyle='--', label='Reconstructed')
    ax.set_ylabel('Amplitude')
    ax.set_title(lbl)
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Section 1: Ridge Reconstruction vs Original', fontsize=13)
plt.tight_layout()
plt.show()

# Reconstruction fidelity (correlation)
for i, (orig, comp) in enumerate(zip(originals, comps)):
    orig_np = orig if isinstance(orig, np.ndarray) else orig.numpy()
    comp_np = np.real(comp if isinstance(comp, np.ndarray) else comp.numpy())
    corr = np.corrcoef(orig_np, comp_np)[0, 1]
    print(f'Ridge {i+1} reconstruction correlation: {corr:.4f}')

## Section 2: Component Onset Detection

`detect_onsets()` finds when a ridge component is active by thresholding its `energy_path`.

Here we synthesize a **gated 40 Hz tone** that is silent outside [0.5, 1.5] s, then verify that
`OnsetResult.t_start` and `t_end` recover the gate boundaries.

In [ ]:
T2  = 2.0
t2  = np.arange(int(T2 * FS)) / FS

# Gated tone: 40 Hz, active only in [0.5, 1.5] s
x_gate = wavesst.make_chirp(f_start=40.0, f_end=40.0, duration=T2, fs=FS,
                             t_start=0.5, t_end=1.5)

fig, ax = plt.subplots(figsize=(10, 2))
ax.plot(t2, x_gate if isinstance(x_gate, np.ndarray) else x_gate.numpy(),
        linewidth=0.7, color='steelblue')
ax.axvline(0.5, color='red', linestyle='--', label='Gate on')
ax.axvline(1.5, color='green', linestyle='--', label='Gate off')
ax.set_xlabel('Time (s)')
ax.set_title('Gated 40 Hz tone')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
result2 = wavesst.sst(x_gate, fs=FS, nv=32, gamma='auto', cfg=cfg)
ridge2  = wavesst.extract_ridges(result2, n=1, penalty=1.0)[0]

onset = wavesst.detect_onsets(ridge2, threshold=0.05)
print(f'Detected onset  : t_start = {onset.t_start:.3f} s  (true = 0.500 s)')
print(f'Detected offset : t_end   = {onset.t_end:.3f} s  (true = 1.500 s)')

In [ ]:
energy_path = ridge2.energy_path if isinstance(ridge2.energy_path, np.ndarray) \
              else ridge2.energy_path.numpy()
times2 = ridge2.times if isinstance(ridge2.times, np.ndarray) else ridge2.times.numpy()

thresh_abs = onset.threshold if hasattr(onset, 'threshold') \
             else float(np.max(energy_path) * 0.05)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(times2, energy_path, color='steelblue', label='Ridge energy path')
ax.axhline(thresh_abs, color='orange', linestyle='--', label=f'Threshold')
ax.axvline(onset.t_start, color='red',   linestyle='-',  linewidth=2, label=f't_start = {onset.t_start:.3f} s')
ax.axvline(onset.t_end,   color='green', linestyle='-',  linewidth=2, label=f't_end   = {onset.t_end:.3f} s')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Energy')
ax.set_title('Section 2: Onset Detection via Energy Thresholding')
ax.legend()
plt.tight_layout()
plt.show()

## Section 3: Masked Ridge Extraction

When two components overlap in energy, standard `extract_ridges` may track the dominant one for
all requested ridges.  The **peel-off** strategy:
1. Extract the first ridge normally.
2. Build a mask that zeros out a ±bandwidth band around that ridge.
3. Call `extract_ridges_masked` to find the next component in the remaining TF plane.

In [ ]:
# Two simultaneous tones: 30 Hz (weak) and 80 Hz (strong)
x_tones = (wavesst.make_chirp(f_start=30.0, f_end=30.0, duration=T2, fs=FS)
          + 2.0 * wavesst.make_chirp(f_start=80.0, f_end=80.0, duration=T2, fs=FS))

result3 = wavesst.sst(x_tones, fs=FS, nv=32, gamma='auto', cfg=cfg)

# First extraction: finds the dominant (80 Hz) component
ridge3_first = wavesst.extract_ridges(result3, n=1, penalty=1.0)[0]
print(f'First ridge median frequency: {np.median(ridge3_first.freq_path):.1f} Hz')

In [ ]:
def make_ridge_mask(sst_result, ridge, bandwidth_hz=10.0):
    """Return a bool mask with False in a bandwidth around the ridge."""
    freqs  = sst_result.freqs if isinstance(sst_result.freqs, np.ndarray) \
             else sst_result.freqs.numpy()
    n_freqs = len(freqs)
    n_times = len(sst_result.times)
    mask = np.ones((n_freqs, n_times), dtype=bool)
    fp = ridge.freq_path if isinstance(ridge.freq_path, np.ndarray) \
         else ridge.freq_path.numpy()
    for t_idx in range(n_times):
        f_center = fp[t_idx]
        for f_idx, f in enumerate(freqs):
            if abs(f - f_center) < bandwidth_hz:
                mask[f_idx, t_idx] = False
    return mask

mask3 = make_ridge_mask(result3, ridge3_first, bandwidth_hz=10.0)
print(f'Mask zeros: {(~mask3).sum()} / {mask3.size} bins suppressed')

ridge3_second = wavesst.extract_ridges_masked(result3, mask3, n=1, penalty=1.0)[0]
print(f'Second ridge median frequency: {np.median(ridge3_second.freq_path):.1f} Hz')

In [ ]:
Tx_mag = result3.Tx.abs().numpy() if hasattr(result3.Tx, 'numpy') else np.abs(result3.Tx)
freqs3 = result3.freqs if isinstance(result3.freqs, np.ndarray) else result3.freqs.numpy()
times3 = result3.times if isinstance(result3.times, np.ndarray) else result3.times.numpy()

fp1 = ridge3_first.freq_path  if isinstance(ridge3_first.freq_path, np.ndarray)  else ridge3_first.freq_path.numpy()
fp2 = ridge3_second.freq_path if isinstance(ridge3_second.freq_path, np.ndarray) else ridge3_second.freq_path.numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.pcolormesh(times3, freqs3, Tx_mag, shading='auto', cmap='inferno')
ax.plot(times3, fp1, color='cyan',  linewidth=2, label=f'Ridge 1 (~{np.median(fp1):.0f} Hz)')
ax.plot(times3, fp2, color='lime',  linewidth=2, label=f'Ridge 2 (~{np.median(fp2):.0f} Hz)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Section 3: Masked Ridge Extraction — Peel-Off Strategy')
ax.legend()
plt.tight_layout()
plt.show()

## Section 4: Parallel Band-Decomposed Extraction

`extract_ridges_parallel` divides the frequency axis into `n` bands and extracts one ridge per band
in parallel, then merges results by energy.  This is faster on multi-core systems and can avoid
ridge-crossing ambiguity by construction.

We compare it against sequential `extract_ridges` on a 3-component signal.

In [ ]:
# Three simultaneous tones: 20, 50, 90 Hz
x3 = (wavesst.make_chirp(f_start=20.0, f_end=20.0, duration=T2, fs=FS)
    + wavesst.make_chirp(f_start=50.0, f_end=50.0, duration=T2, fs=FS)
    + wavesst.make_chirp(f_start=90.0, f_end=90.0, duration=T2, fs=FS))

result4 = wavesst.sst(x3, fs=FS, nv=32, gamma='auto', cfg=cfg)
print('SST computed.')

In [ ]:
import time

t0 = time.perf_counter()
ridges_seq = wavesst.extract_ridges(result4, n=3, penalty=1.0)
dt_seq = time.perf_counter() - t0

t0 = time.perf_counter()
ridges_par = wavesst.extract_ridges_parallel(result4, n=3, penalty=1.0, n_jobs=-1)
dt_par = time.perf_counter() - t0

print(f'Sequential : {dt_seq*1e3:.1f} ms')
print(f'Parallel   : {dt_par*1e3:.1f} ms')

print('\nSequential ridge medians (Hz):', sorted([np.median(r.freq_path) for r in ridges_seq]))
print('Parallel   ridge medians (Hz):', sorted([np.median(r.freq_path) for r in ridges_par]))

In [ ]:
Tx4   = result4.Tx.abs().numpy() if hasattr(result4.Tx, 'numpy') else np.abs(result4.Tx)
f4    = result4.freqs if isinstance(result4.freqs, np.ndarray) else result4.freqs.numpy()
t4    = result4.times if isinstance(result4.times, np.ndarray) else result4.times.numpy()

colors = ['cyan', 'lime', 'yellow']

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, ridges, title in zip(axes,
                              [ridges_seq, ridges_par],
                              ['Sequential extract_ridges', 'Parallel extract_ridges_parallel']):
    ax.pcolormesh(t4, f4, Tx4, shading='auto', cmap='inferno')
    for r, c in zip(sorted(ridges, key=lambda r: np.median(r.freq_path)), colors):
        fp = r.freq_path if isinstance(r.freq_path, np.ndarray) else r.freq_path.numpy()
        ax.plot(t4, fp, color=c, linewidth=2, label=f'~{np.median(fp):.0f} Hz')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)
    ax.legend()

plt.suptitle('Section 4: Sequential vs Parallel Ridge Extraction', fontsize=13)
plt.tight_layout()
plt.show()